# 24 gprofiler validation

**Purpose:** Validate selected gene lists through the g:Profiler API.

**Original analysis notebook:** `22b_online_enrichment_validation_HVG8000_r0_6_v2_no_pip_direct_api.ipynb`

Run this notebook from top to bottom in a fresh kernel. Project paths are
resolved from `GLIOMA_PROJECT_ROOT`; when that variable is not set, the
notebook searches parent directories for the repository root. Generated
AnnData objects are written under `data/processed/` or `data/interim/`,
while tables, figures, and logs retain the workflow's historical output
directories at the repository root.

## Online validation scope

Selected state-marker and condition-level malignant DE lists containing at
least 10 genes are submitted to the g:Profiler g:GOSt API. Enrichr is not
used in the submitted workflow. API failures are recorded per gene list so
that a transient network error does not erase successful results from other
lists.

In [ ]:
import json
import math
import os
import time
import urllib.error
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

def _find_repository_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start

PROJECT_DIR = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", _find_repository_root())
).expanduser().resolve()
METADATA_DIR = PROJECT_DIR / "metadata"
METADATA_DIR.mkdir(parents=True, exist_ok=True)

OUT_PREFIX = "online_enrichment_validation_HVG8000_r0_6_gprofiler_only"
STATE_SIG_TSV = METADATA_DIR / "malignant_state_pseudobulk_DE_state_vs_rest_significant_HVG8000_r0_6.tsv"
COND_SIG_TSV = METADATA_DIR / "condition_malignant_pseudobulk_DE_HVG8000_r0_6_overall_malignant_pairwise_DE_significant_FDR0_05_absLFC0_5.tsv"

MAX_GENES_PER_LIST = 300
MIN_GENES_PER_LIST = 10
REQUEST_SLEEP_SEC = 1.0
REQUEST_TIMEOUT_SEC = 180
MAX_RETRIES = 3
GPROFILER_SOURCES = ["GO:BP", "REAC", "KEGG", "WP"]

def write_tsv(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, sep="\t", index=False)
    print("[SAVED]", path, "shape=", df.shape)
    return path

for required in (STATE_SIG_TSV, COND_SIG_TSV):
    if not required.exists():
        raise FileNotFoundError(f"Required DE table not found: {required}")

In [ ]:
state_sig = pd.read_csv(STATE_SIG_TSV, sep="\t")
cond_sig = pd.read_csv(COND_SIG_TSV, sep="\t")

def clean_gene(value):
    if pd.isna(value):
        return None
    value = str(value).strip()
    return None if value.lower() in {"", "nan", "none"} else value

def unique_genes(values):
    output, seen = [], set()
    for value in values:
        gene = clean_gene(value)
        if gene and gene not in seen:
            output.append(gene)
            seen.add(gene)
    return output

gene_lists = {}
inventory_rows = []

state_sort = "padj_fdr_bh" if "padj_fdr_bh" in state_sig else "pval_mannwhitney"
for state, sub in state_sig.sort_values(["malignant_state", state_sort]).groupby("malignant_state"):
    up = sub[sub["direction"].astype(str).str.contains("up", case=False, na=False)]
    genes = unique_genes(up["gene"].head(MAX_GENES_PER_LIST))
    name = f"state_marker__{state}__up"
    if len(genes) >= MIN_GENES_PER_LIST:
        gene_lists[name] = genes
        inventory_rows.append({"gene_list": name, "source": "malignant_state_marker_DE", "group": state, "direction": "up", "n_genes": len(genes)})

condition_sort = "padj_welch" if "padj_welch" in cond_sig else ("padj_mannwhitney" if "padj_mannwhitney" in cond_sig else None)
ordered_condition = cond_sig.sort_values(["contrast", condition_sort]) if condition_sort else cond_sig
for contrast, sub in ordered_condition.groupby("contrast"):
    condition_a = str(sub["condition_A"].iloc[0])
    condition_b = str(sub["condition_B"].iloc[0])
    fold_change = pd.to_numeric(sub["log2FC_A_vs_B"], errors="coerce")
    for direction, selected in [
        (f"{condition_a}_up", sub[fold_change > 0]),
        (f"{condition_b}_up", sub[fold_change < 0]),
    ]:
        genes = unique_genes(selected["gene"].head(MAX_GENES_PER_LIST))
        name = f"condition_overall__{contrast}__{direction}"
        if len(genes) >= MIN_GENES_PER_LIST:
            gene_lists[name] = genes
            inventory_rows.append({"gene_list": name, "source": "condition_overall_malignant_DE", "group": contrast, "direction": direction, "n_genes": len(genes)})

inventory = pd.DataFrame(inventory_rows).sort_values(["source", "gene_list"])
write_tsv(inventory, METADATA_DIR / f"{OUT_PREFIX}_gene_list_inventory.tsv")
print("Gene lists prepared:", len(gene_lists))
display(inventory)

In [ ]:
def post_json_with_retries(url, payload):
    encoded = json.dumps(payload).encode("utf-8")
    last_error = None
    for attempt in range(1, MAX_RETRIES + 1):
        request = urllib.request.Request(
            url,
            data=encoded,
            headers={
                "Content-Type": "application/json",
                "User-Agent": "glioma-cnv-single-cell-atlas/1.0",
            },
        )
        try:
            with urllib.request.urlopen(request, timeout=REQUEST_TIMEOUT_SEC) as response:
                return json.loads(response.read().decode("utf-8"))
        except Exception as exc:
            last_error = exc
            if attempt < MAX_RETRIES:
                time.sleep(2 ** (attempt - 1))
    raise RuntimeError(f"g:Profiler request failed after {MAX_RETRIES} attempts: {last_error!r}")

def run_gprofiler(list_name, genes):
    response = post_json_with_retries(
        "https://biit.cs.ut.ee/gprofiler/api/gost/profile/",
        {
            "organism": "hsapiens",
            "query": genes,
            "sources": GPROFILER_SOURCES,
            "user_threshold": 0.05,
            "all_results": False,
            "no_evidences": True,
            "ordered": False,
            "significance_threshold_method": "g_SCS",
        },
    )
    rows = []
    for result in response.get("result", []):
        p_value = result.get("p_value")
        rows.append({
            "gene_list": list_name,
            "source": result.get("source"),
            "term_id": result.get("native"),
            "term_name": result.get("name"),
            "p_value": p_value,
            "negative_log10_p": -math.log10(p_value) if p_value and p_value > 0 else np.inf,
            "term_size": result.get("term_size"),
            "query_size": result.get("query_size"),
            "intersection_size": result.get("intersection_size"),
            "precision": result.get("precision"),
            "recall": result.get("recall"),
            "intersections": ";".join(result.get("intersections", []) or []),
        })
    return rows

result_rows, error_rows = [], []
for number, (name, genes) in enumerate(gene_lists.items(), start=1):
    print(f"[g:Profiler] {number}/{len(gene_lists)} {name} n={len(genes)}")
    try:
        result_rows.extend(run_gprofiler(name, genes))
    except Exception as exc:
        error_rows.append({"gene_list": name, "n_genes": len(genes), "error": repr(exc)})
        print("[ERROR]", name, repr(exc))
    time.sleep(REQUEST_SLEEP_SEC)

result_columns = ["gene_list", "source", "term_id", "term_name", "p_value", "negative_log10_p", "term_size", "query_size", "intersection_size", "precision", "recall", "intersections"]
results = pd.DataFrame(result_rows, columns=result_columns)
errors = pd.DataFrame(error_rows, columns=["gene_list", "n_genes", "error"])
top_terms = results.sort_values(["gene_list", "p_value"]).groupby("gene_list", group_keys=False).head(10) if len(results) else results.copy()

write_tsv(results, METADATA_DIR / f"{OUT_PREFIX}_gprofiler_results.tsv")
write_tsv(top_terms, METADATA_DIR / f"{OUT_PREFIX}_gprofiler_top_terms.tsv")
write_tsv(errors, METADATA_DIR / f"{OUT_PREFIX}_gprofiler_errors.tsv")
display(results.head(20))

In [ ]:
CATEGORY_KEYWORDS = {
    "hypoxia_glycolysis_stress": ["hypoxia", "hif", "glycolysis", "glucose", "lactate", "oxidative stress", "unfolded protein", "stress"],
    "cycling_E2F_G2M": ["cell cycle", "mitotic", "mitosis", "g2m", "e2f", "dna replication", "chromosome segregation", "spindle"],
    "MES_ECM_remodeling_invasion": ["extracellular matrix", "collagen", "matrix", "focal adhesion", "migration", "invasion", "wound"],
    "OPC_oligodendrocyte_glial": ["oligodendrocyte", "glial", "gliogenesis", "myelination", "axon ensheathment"],
    "neural_NPC_development": ["neuron", "neural", "neurogenesis", "axon", "synapse", "dendrite", "forebrain", "nervous system development"],
    "immune_inflammation_interferon_TNF": ["immune", "inflammatory", "interferon", "cytokine", "tnf", "nf-kappa", "leukocyte", "antigen"],
    "angiogenesis_endothelial": ["angiogenesis", "endothelial", "blood vessel", "vasculature", "vascular"],
    "dna_damage_repair": ["dna repair", "dna damage", "double-strand", "replication stress", "checkpoint"],
}

def categorize_term(term):
    text = str(term).lower()
    categories = [category for category, keywords in CATEGORY_KEYWORDS.items() if any(keyword in text for keyword in keywords)]
    return ";".join(categories) if categories else "other"

summary_rows = []
if len(results):
    categorized = results.copy()
    categorized["category"] = categorized["term_name"].map(categorize_term)
    categorized = categorized[categorized["category"] != "other"]
    for (gene_list, category), sub in categorized.groupby(["gene_list", "category"]):
        summary_rows.append({
            "source": "g:Profiler",
            "gene_list": gene_list,
            "category": category,
            "n_terms": len(sub),
            "best_p_value": sub["p_value"].min(),
            "top_terms": " | ".join(sub.sort_values("p_value")["term_name"].head(5).astype(str)),
        })

biological_summary = pd.DataFrame(summary_rows, columns=["source", "gene_list", "category", "n_terms", "best_p_value", "top_terms"])
category_counts = biological_summary.groupby(["source", "category"]).size().reset_index(name="n_gene_list_category_hits") if len(biological_summary) else pd.DataFrame(columns=["source", "category", "n_gene_list_category_hits"])

write_tsv(biological_summary, METADATA_DIR / f"{OUT_PREFIX}_biological_keyword_summary.tsv")
write_tsv(category_counts, METADATA_DIR / f"{OUT_PREFIX}_biological_category_counts.tsv")

output_names = [
    f"{OUT_PREFIX}_gene_list_inventory.tsv",
    f"{OUT_PREFIX}_gprofiler_results.tsv",
    f"{OUT_PREFIX}_gprofiler_top_terms.tsv",
    f"{OUT_PREFIX}_gprofiler_errors.tsv",
    f"{OUT_PREFIX}_biological_keyword_summary.tsv",
    f"{OUT_PREFIX}_biological_category_counts.tsv",
]
manifest = pd.DataFrame([
    {
        "file": name,
        "path": str(METADATA_DIR / name),
        "exists": (METADATA_DIR / name).exists(),
        "size_bytes": (METADATA_DIR / name).stat().st_size if (METADATA_DIR / name).exists() else 0,
    }
    for name in output_names
])
write_tsv(manifest, METADATA_DIR / f"{OUT_PREFIX}_output_manifest.tsv")
display(manifest)
print("[DONE] g:Profiler validation")